# Day 3 - Homework: Market basket on real transactions

**Goal:** run the exact technique from Lab 3 on data that is *natively* transactional, one row per (basket, item), and see how much stronger the rules become. This is market basket analysis working with the grain of the data rather than against it.

Keep Lab 3 in mind: there, the strongest rule from constructed portal baskets barely reached a lift of 1.1. Watch what happens here.

In [ ]:
import importlib, importlib.util, subprocess, sys

def ensure(pkg, import_name=None):
    """Install pkg if absent, surface any pip error, and confirm it imports."""
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        print(f'installing {pkg} ...')
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout[-2000:]); print(r.stderr[-2000:])
            raise RuntimeError(f'Could not install {pkg}. See the pip output above.')
        importlib.invalidate_caches()
    importlib.import_module(name)
    print(f'{pkg} ready')

ensure('mlxtend')

import pandas as pd
from pathlib import Path

def find_data(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / 'data').is_dir():
            return p / 'data'
    raise FileNotFoundError('data/ folder not found')

DATA = find_data()
tx = pd.read_csv(DATA / 'retail' / 'retail_transactions.csv')
print('rows (basket-item pairs):', len(tx))
print('baskets:', tx['basket_id'].nunique(), ' distinct items:', tx['item'].nunique())
tx.head()

## 1. Shape into transactions

The file is long form: one row per item in a basket. Market basket wants each basket as a *set* of items.

In [ ]:
# TODO: turn the long-form table into one row per basket: group tx by 'basket_id'
# and collect each basket's items into a list. Print an example basket and the
# average basket size.


## 2. Frequent itemsets and rules

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# TODO: encode `baskets` with TransactionEncoder into a one-hot DataFrame, run
# apriori(min_support=0.03, use_colnames=True), then association_rules(metric='lift',
# min_threshold=1.0) sorted by lift descending into a variable called `rules`.
# Print how many frequent itemsets and rules you found.


## 3. Read the strongest rules

Three numbers describe every rule. **Support**: how common the combination is overall. **Confidence**: given the left-hand items, how often the right-hand items appear. **Lift**: how many times more likely that is than chance. Lift above 1 means a genuine association; the higher, the stronger.

In [ ]:
top = rules.head(8).copy()
top['antecedents'] = top['antecedents'].apply(lambda s: ', '.join(s))
top['consequents'] = top['consequents'].apply(lambda s: ', '.join(s))
print(top[['antecedents', 'consequents', 'support', 'confidence', 'lift']].to_string(index=False))

These lifts sit far above 1, several times stronger than anything the portal baskets could produce. Same algorithm, same three lines of mlxtend. The only thing that changed is that this data is *actually* transactional. That contrast is the whole point of the exercise.

## 4. From rule to decision

A strong rule is not automatically a useful one. The skill market basket analysis really trains is asking the right business question of a rule: is it *actionable*, and is it *causal enough to act on*?

Retail folklore is full of 'surprising' pairings, two apparently unrelated products that supposedly sell together, stories retold as tidy causal wins when they were nothing of the sort. That is the right instinct to carry: a high lift tells you two things co-occur, not *why*. Before acting, ask whether the association would survive being turned into a shelf layout, a bundle or a recommendation, or whether it is just two popular items that happen to share baskets.

## Your turn

1. Pick the single rule you would actually act on if you ran this shop, and justify it using support, confidence and lift together rather than lift alone.
2. Raise `min_support` to 0.10 and rerun. Which rules survive, and what kind of association does a high support threshold bias you towards?
3. In two sentences, explain to a non-technical manager why the portal 'baskets' in Lab 3 gave such weak rules while this dataset gives strong ones.

In [ ]:
# your turn
